# 🧬 Gemma-4-12B Bioinformatics Fine-Tuning — FINAL CLEAN VERSION
**Environment:** Python 3.11 · `gemma4` conda env  
**Hardware:** NVIDIA RTX 4500 Ada Generation (24GB VRAM) — GPU 0  
**Strategy:** QLoRA (4-bit NF4) + SFT via TRL  
**Model:** `google/gemma-4-12B-it`  
**Dataset:** `yashm/bioinformatics-qa-dataset`

---
### ⚠️ Pre-flight checklist
- [ ] Kernel set to **Python 3.11 (gemma4)**
- [ ] `HF_TOKEN` updated in Cell 3
- [ ] Run cells **top to bottom, in order**

In [ ]:
# ============================================================
# CELL 1 — PIN GPU + ENVIRONMENT
# MUST be first cell executed — sets env before torch imports
# ============================================================
import os

os.environ["CUDA_VISIBLE_DEVICES"]    = "0"
os.environ["TOKENIZERS_PARALLELISM"]  = "false"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = (
    "max_split_size_mb:128,garbage_collection_threshold:0.8"
)

print("✅ GPU pinned to device 0")
print(f"   CUDA_VISIBLE_DEVICES   = {os.environ['CUDA_VISIBLE_DEVICES']}")
print(f"   TOKENIZERS_PARALLELISM = {os.environ['TOKENIZERS_PARALLELISM']}")

In [ ]:
# ============================================================
# CELL 2 — ENVIRONMENT VERIFICATION
# Confirms Python 3.11, gemma4 conda env, all packages,
# CUDA access, and gemma4_unified architecture support.
# No installs — just checks. Fix issues before continuing.
# ============================================================
import sys, importlib

print("=" * 58)
print("  ENVIRONMENT VERIFICATION")
print("=" * 58)
print(f"  Python  : {sys.version.split()[0]}")
print(f"  Env     : {sys.prefix}")

# Must be Python 3.11+
assert sys.version_info >= (3, 11), (
    f"❌ Wrong Python: {sys.version}\n"
    "   Switch kernel → Kernel → Change Kernel → Python 3.11 (gemma4)"
)
print(f"  ✅ Python 3.11+ confirmed\n")

# Package versions
from packaging.version import Version
required = {
    "torch"          : "2.0.0",
    "transformers"   : "5.0.0",
    "trl"            : "0.12.0",
    "peft"           : "0.14.0",
    "accelerate"     : "1.2.0",
    "bitsandbytes"   : "0.45.0",
    "datasets"       : "3.0.0",
}

all_ok = True
print(f"  {'Package':<18} {'Version':<22} Status")
print("  " + "-" * 52)
for pkg, min_ver in required.items():
    try:
        mod = importlib.import_module(pkg)
        ver = getattr(mod, "__version__", "installed")
        try:
            ok = Version(ver) >= Version(min_ver)
        except Exception:
            ok = True
        flag = "✅" if ok else f"⚠️  need >={min_ver}"
        if not ok:
            all_ok = False
        print(f"  {pkg:<18} {ver:<22} {flag}")
    except ImportError:
        print(f"  {pkg:<18} {'NOT FOUND':<22} ❌")
        all_ok = False

# liger_kernel has no __version__
try:
    import liger_kernel
    print(f"  {'liger_kernel':<18} {'installed':<22} ✅")
except ImportError:
    print(f"  {'liger_kernel':<18} {'NOT FOUND':<22} ❌")
    all_ok = False

# CUDA check
import torch
print(f"\n  {'─'*52}")
print(f"  CUDA available : {torch.cuda.is_available()}")
assert torch.cuda.is_available(), "❌ CUDA not found!"
print(f"  CUDA version   : {torch.version.cuda}")
print(f"  GPU 0          : {torch.cuda.get_device_name(0)}")
print(f"  VRAM           : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")

# gemma4_unified support
from transformers.models.auto.configuration_auto import CONFIG_MAPPING
g4_ok = "gemma4_unified" in CONFIG_MAPPING
print(f"\n  gemma4_unified : {'✅ Recognized' if g4_ok else '❌ NOT FOUND — reinstall transformers from git'}")
if not g4_ok:
    all_ok = False

print(f"\n{'='*58}")
print(f"  {'✅ ALL CHECKS PASSED — proceed to Cell 3' if all_ok else '❌ Fix issues above before continuing'}")
print(f"{'='*58}")

In [ ]:
# ============================================================
# CELL 3 — MASTER IMPORTS + GLOBAL CONFIG
# Re-run this cell after ANY kernel restart before
# running any other cell.
# ============================================================
import gc, time, json
import torch
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForImageTextToText,   # ← Gemma-4 is multimodal
    BitsAndBytesConfig,
)
from peft import LoraConfig, TaskType, PeftModel
from trl import SFTTrainer, SFTConfig
from huggingface_hub import login, whoami, model_info

# ── 🔑 UPDATE THIS ───────────────────────────────────────────
HF_TOKEN     = "hf_XXXXXXXXXXXXXXXXXXXXXXXXXXXXXX"   # ← your token

# ── Model & Dataset ──────────────────────────────────────────
MODEL_ID     = "google/gemma-4-12B-it"
DATASET_ID   = "yashm/bioinformatics-qa-dataset"
OUTPUT_DIR   = "./gemma4-12b-bioinfo-qlora"

# ── Training constants (tuned for RTX 4500 Ada 24GB) ─────────
MAX_SEQ_LEN  = 1024
BATCH_SIZE   = 1
GRAD_ACCUM   = 16       # effective batch = 1 × 16 = 16
LORA_R       = 16
LORA_ALPHA   = 32

SYSTEM_PROMPT = (
    "You are an expert bioinformatics assistant with deep knowledge of "
    "genomics, proteomics, transcriptomics, sequence analysis, "
    "biological databases (UniProt, NCBI, Ensembl, PDB), and tools "
    "(BLAST, HMMER, BWA, GATK, DESeq2, Bowtie2, STAR, Samtools, BioPython). "
    "Provide accurate, concise, and scientifically rigorous answers."
)

# ── Authenticate ─────────────────────────────────────────────
login(token=HF_TOKEN, add_to_git_credential=False)
user = whoami()
print(f"✅ Logged in as : {user['name']}")

# ── Confirm model access ─────────────────────────────────────
info = model_info(MODEL_ID, token=HF_TOKEN)
print(f"✅ Model access : GRANTED ({info.id})")

# ── GPU setup ────────────────────────────────────────────────
assert torch.cuda.is_available(), "❌ No CUDA GPU found!"
gpu        = torch.cuda.get_device_properties(0)
TOTAL_VRAM = gpu.total_memory / 1024**3

def vram(label=""):
    alloc    = torch.cuda.memory_allocated(0) / 1024**3
    reserved = torch.cuda.memory_reserved(0)  / 1024**3
    free     = TOTAL_VRAM - reserved
    print(f"  📊 VRAM [{label}] — "
          f"allocated: {alloc:.2f}GB | "
          f"reserved: {reserved:.2f}GB | "
          f"free: {free:.2f}GB")

import transformers
print(f"\n✅ Stack ready:")
print(f"   Python        : {__import__('sys').version.split()[0]}")
print(f"   PyTorch       : {torch.__version__}")
print(f"   transformers  : {transformers.__version__}")
print(f"   GPU           : {gpu.name} ({TOTAL_VRAM:.1f}GB)")
vram("baseline")

In [ ]:
# ============================================================
# CELL 4 — LOAD & INSPECT DATASET
# ============================================================
print(f"📥 Loading: {DATASET_ID}")
raw_dataset = load_dataset(DATASET_ID, token=HF_TOKEN)

print("\n📦 Dataset loaded:")
print(raw_dataset)
print(f"\n📋 Columns   : {raw_dataset['train'].column_names}")
print(f"   Train rows : {len(raw_dataset['train']):,}")
if "test" in raw_dataset:
    print(f"   Test rows  : {len(raw_dataset['test']):,}")

print("\n── First sample ──────────────────────────────────────")
for k, v in raw_dataset["train"][0].items():
    print(f"  [{k}]: {str(v)[:250]}")

In [ ]:
# ============================================================
# CELL 5 — FORMAT DATASET TO GEMMA-4 CHAT FORMAT
#
# ⚠️  Check Cell 4 output and update column names below
#     if your dataset uses different names.
#
# IMPORTANT: Gemma-4 uses role "model" NOT "assistant"
# ============================================================

# ── Update if Cell 4 shows different column names ────────────
QUESTION_COL = "question"   # ← change if needed
ANSWER_COL   = "answer"     # ← change if needed

cols = raw_dataset["train"].column_names
assert QUESTION_COL in cols, (
    f"❌ '{QUESTION_COL}' not found. Available: {cols}"
)
assert ANSWER_COL in cols, (
    f"❌ '{ANSWER_COL}' not found. Available: {cols}"
)
print(f"✅ Columns confirmed: '{QUESTION_COL}' + '{ANSWER_COL}'")

def format_to_messages(example):
    """Convert Q&A → Gemma-4 chat messages.
    Note: Gemma-4 uses 'model' role (not 'assistant') for responses.
    """
    return {
        "messages": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user",   "content": str(example[QUESTION_COL]).strip()},
            {"role": "model",  "content": str(example[ANSWER_COL]).strip()},
        ]
    }

formatted_dataset = raw_dataset.map(
    format_to_messages,
    remove_columns=raw_dataset["train"].column_names,
    num_proc=4,
    desc="Formatting to Gemma-4 chat format",
)

# Create eval split if none exists
if "test" not in formatted_dataset and "validation" not in formatted_dataset:
    print("ℹ️  No eval split — creating 95/5 split")
    from datasets import DatasetDict
    split = formatted_dataset["train"].train_test_split(
        test_size=0.05, seed=42
    )
    formatted_dataset = DatasetDict({
        "train": split["train"],
        "test" : split["test"]
    })

EVAL_KEY = "test" if "test" in formatted_dataset else "validation"

print(f"\n✅ Dataset formatted")
print(f"   Train : {len(formatted_dataset['train']):,}")
print(f"   Eval  : {len(formatted_dataset[EVAL_KEY]):,}")

print("\n── Sample messages ────────────────────────────────────")
for msg in formatted_dataset["train"][0]["messages"]:
    content = msg["content"]
    print(f"  [{msg['role'].upper():9s}]: {content[:150]}"
          f"{'...' if len(content) > 150 else ''}")

In [ ]:
# ============================================================
# CELL 6 — QLORA CONFIGURATION
# Tuned for RTX 4500 Ada 24GB single GPU
# ============================================================

# 4-bit NF4 quantization: shrinks 12B model from ~24GB → ~6GB
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,       # extra ~0.5GB savings
    bnb_4bit_quant_storage=torch.bfloat16,
)

# LoRA adapter: ~0.3GB overhead, targets all linear layers
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=0.05,
    bias="none",
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",   # attention
        "gate_proj", "up_proj", "down_proj",        # MLP/SwiGLU
    ],
)

print("✅ QLoRA configuration ready")
print(f"   Quantization   : 4-bit NF4 + double quant")
print(f"   LoRA rank      : {lora_config.r}")
print(f"   LoRA alpha     : {lora_config.lora_alpha}")
print(f"   Target modules : {lora_config.target_modules}")
print(f"\n   📊 Estimated VRAM usage:")
print(f"     Model (4-bit NF4 + dq)   : ~6.0 GB")
print(f"     LoRA adapters (r=16)     : ~0.3 GB")
print(f"     Gradient checkpointing   : ~3.5 GB")
print(f"     Paged AdamW optimizer    : ~1.5 GB")
print(f"     CUDA overhead + buffers  : ~2.0 GB")
print(f"     ─────────────────────────────────────")
print(f"     Total estimated peak     : ~13 GB ✅")
print(f"     Headroom (on 24GB)       : ~11 GB ✅")

In [ ]:
# ============================================================
# CELL 7 — LOAD TOKENIZER
# Clean load — no patches needed with Python 3.11 +
# transformers 5.x from git source
# ============================================================
print(f"📥 Loading tokenizer: {MODEL_ID}")

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    token=HF_TOKEN,
    trust_remote_code=True,
)

# Gemma-4 has <pad> token (id=0) — use it directly
if tokenizer.pad_token is None:
    tokenizer.pad_token    = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id

tokenizer.padding_side = "right"   # right-pad during training

print(f"\n✅ Tokenizer loaded")
print(f"   Vocab size    : {tokenizer.vocab_size:,}")
print(f"   EOS token     : {tokenizer.eos_token!r}  "
      f"(id={tokenizer.eos_token_id})")
print(f"   PAD token     : {tokenizer.pad_token!r}  "
      f"(id={tokenizer.pad_token_id})")
print(f"   Padding side  : {tokenizer.padding_side}")
print(f"   Chat template : "
      f"{'✅ Present' if tokenizer.chat_template else '❌ MISSING!'}")

# Verify with Gemma-4 role names
# ⚠️  Gemma-4 uses 'model' role (NOT 'assistant')
test_msgs = [
    {"role": "system", "content": "You are a bioinformatics expert."},
    {"role": "user",   "content": "What is BLAST?"},
    {"role": "model",  "content": "BLAST is a sequence alignment tool."},
]
rendered = tokenizer.apply_chat_template(
    test_msgs, tokenize=False, add_generation_prompt=False
)
print(f"\n── Chat template preview ──────────────────────────────")
print(rendered)
print("✅ Chat template works correctly")

In [ ]:
# ============================================================
# CELL 8 — LOAD GEMMA-4-12B IN 4-BIT QLORA
#
# Key changes from generic notebooks:
#   ✅ AutoModelForImageTextToText  (gemma4_unified is multimodal)
#   ✅ dtype=  not torch_dtype=     (fixes deprecation warning)
#   ✅ text_config for architecture info (top-level config has none)
# ============================================================
gc.collect()
torch.cuda.empty_cache()
vram("before model load")

print(f"\n📥 Loading: {MODEL_ID}")
print("   Architecture  : gemma4_unified (text + vision)")
print("   Loader        : AutoModelForImageTextToText")
print("   Quantization  : 4-bit NF4")
print("   Device        : GPU 0")
print("   ⏳ First run downloads ~7GB (~5-10 min)...\n")

model = AutoModelForImageTextToText.from_pretrained(
    MODEL_ID,
    token=HF_TOKEN,
    quantization_config=bnb_config,
    device_map={"" : 0},                  # pin everything to GPU 0
    attn_implementation="eager",           # required for Gemma-4
    dtype=torch.bfloat16,                  # ← dtype= (not torch_dtype=)
    trust_remote_code=True,
)

model.config.use_cache = False
model.enable_input_require_grads()

total_params = sum(p.numel() for p in model.parameters())
vram("after model load")

print(f"\n✅ Model loaded!")
print(f"   model_type   : {model.config.model_type}")
print(f"   total params : {total_params / 1e9:.2f}B")
print(f"   dtype        : {next(model.parameters()).dtype}")
print(f"   device       : {next(model.parameters()).device}")
print(f"   use_cache    : {model.config.use_cache}")
print(f"   attn impl    : {model.config._attn_implementation}")

# Gemma4UnifiedConfig stores text details in text_config
tcfg = model.config.text_config
print(f"\n── Text architecture ──────────────────────────────────")
print(f"   hidden size  : {tcfg.hidden_size}")
print(f"   layers       : {tcfg.num_hidden_layers}")
print(f"   attn heads   : {tcfg.num_attention_heads}")
print(f"   vocab size   : {tcfg.vocab_size:,}")

# Check PEFT state using hasattr
# is_peft_model() was removed from peft 0.19+ public API
has_peft = hasattr(model, "peft_config")
print(f"\n   PEFT applied : {has_peft}  (must be False ✅)")
assert not has_peft, (
    "❌ PEFT already applied — delete model variable "
    "and re-run this cell."
)

In [ ]:
# ============================================================
# CELL 9 — SFT TRAINING CONFIGURATION
#
# Key fix: assistant_only_loss=False
# Reason: Gemma-4 chat template lacks {% generation %} markers
#         TRL cannot auto-patch it for Gemma-4 architecture.
#         Full-sequence loss still trains domain knowledge well.
# ============================================================
sft_config = SFTConfig(
    # ── Output & Logging ────────────────────────────────────
    output_dir=OUTPUT_DIR,
    run_name="gemma4-12b-bioinfo-24gb",
    logging_dir=f"{OUTPUT_DIR}/logs",
    logging_steps=10,
    report_to="none",           # set 'wandb' or 'tensorboard' if desired

    # ── Precision ───────────────────────────────────────────
    bf16=True,                  # RTX 4500 Ada has native bf16 tensor cores
    fp16=False,

    # ── Sequence Length ─────────────────────────────────────
    max_length=MAX_SEQ_LEN,     # 1024 — minimises activation memory

    # ── Batch ───────────────────────────────────────────────
    per_device_train_batch_size=BATCH_SIZE,    # 1
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,   # 16 → eff. batch = 16

    # ── Memory savings ──────────────────────────────────────
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},

    # ── Optimizer ───────────────────────────────────────────
    optim="paged_adamw_8bit",   # pages optimizer states to CPU RAM
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    weight_decay=0.01,
    max_grad_norm=1.0,

    # ── Duration ────────────────────────────────────────────
    num_train_epochs=3,

    # ── Evaluation & Checkpoints ────────────────────────────
    eval_strategy="steps",
    eval_steps=50,
    save_strategy="steps",
    save_steps=50,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,

    # ── SFT specific ────────────────────────────────────────
    dataset_num_proc=4,
    packing=False,

    # ✅ MUST BE FALSE for Gemma-4
    # Gemma-4 chat template missing {% generation %} markers
    # TRL raises ValueError if True with this template
    assistant_only_loss=False,

    # ── Liger kernel ────────────────────────────────────────
    # Fuses RMSNorm + RoPE + SwiGLU → saves ~2GB activation memory
    use_liger_kernel=True,

    # ── Hub (push after training) ────────────────────────────
    push_to_hub=False,
    hub_model_id=HUB_MODEL_ID,
    hub_token=HF_TOKEN,
)

print("✅ SFTConfig ready")
print(f"   assistant_only_loss  : {sft_config.assistant_only_loss} ✅")
print(f"   Effective batch size : {BATCH_SIZE * GRAD_ACCUM}")
print(f"   Max seq length       : {sft_config.max_length}")
print(f"   Optimizer            : {sft_config.optim}")
print(f"   Learning rate        : {sft_config.learning_rate}")
print(f"   Liger kernel         : {sft_config.use_liger_kernel}")
print(f"   Epochs               : {sft_config.num_train_epochs}")

In [ ]:
# ============================================================
# BUILD TRAINER
# ============================================================
print("🔧 Building SFTTrainer with QLoRA adapters...")

# Safety guard: prevent double-PEFT error
# is_peft_model() removed from peft 0.19+ → use hasattr instead
if hasattr(model, "peft_config"):
    raise RuntimeError(
        "❌ Model already has PEFT adapters!\n"
        "   Run: del model  →  gc.collect()  "
        "→  torch.cuda.empty_cache()  →  re-run Cell 8"
    )
print("✅ Model clean — no PEFT adapters yet")

trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=formatted_dataset["train"],
    eval_dataset=formatted_dataset[EVAL_KEY],
    processing_class=tokenizer,    # modern API (replaces tokenizer=)
    peft_config=lora_config,       # injects LoRA into frozen model
)

# Show trainable parameter count
trainer.model.print_trainable_parameters()
vram("after trainer init")

# Show first 6 trainable adapter layers
print("\n── Trainable LoRA adapter layers (first 6) ───────────")
count = 0
for name, param in trainer.model.named_parameters():
    if param.requires_grad:
        print(f"   ✅ {name}  {list(param.shape)}")
        count += 1
        if count >= 6:
            remaining = sum(
                1 for _, p in trainer.model.named_parameters()
                if p.requires_grad
            ) - 6
            print(f"   ... and {remaining} more LoRA layers")
            break

print("\n✅ Trainer ready — run Cell 11 to start training! 🚀")

In [ ]:
# ============================================================
# 🚀 LAUNCH TRAINING
# ============================================================
gc.collect()
torch.cuda.empty_cache()
vram("pre-training")

print("=" * 60)
print("🚀 Starting QLoRA Fine-Tuning")
print(f"   Model   : {MODEL_ID}")
print(f"   Dataset : {DATASET_ID}  ({len(formatted_dataset['train']):,} samples)")
print(f"   GPU     : {gpu.name}")
print(f"   Epochs  : {sft_config.num_train_epochs}")
print(f"   Eff. BS : {BATCH_SIZE * GRAD_ACCUM}")
print(f"   Max len : {MAX_SEQ_LEN}")
print(f"   LR      : {sft_config.learning_rate}")
print("=" * 60)

t_start = time.time()
train_result = trainer.train()
t_elapsed = (time.time() - t_start) / 60

print(f"\n{'='*60}")
print(f"✅ Training complete!")
print(f"   Time         : {t_elapsed:.1f} minutes")
print(f"   Train loss   : {train_result.metrics.get('train_loss', 'N/A'):.4f}")
print(f"   Total steps  : {train_result.global_step}")
print(f"   Samples/sec  : "
      f"{train_result.metrics.get('train_samples_per_second', 'N/A')}")

trainer.log_metrics("train", train_result.metrics)
trainer.save_metrics("train", train_result.metrics)
trainer.save_state()
vram("post-training")

In [ ]:
# ============================================================
# EVALUATE ON EVAL SPLIT
# ============================================================
print("📊 Running evaluation...")

eval_results = trainer.evaluate()

print("\n✅ Evaluation results:")
for k, v in eval_results.items():
    if isinstance(v, float):
        print(f"   {k:35s}: {v:.4f}")
    else:
        print(f"   {k:35s}: {v}")

trainer.log_metrics("eval", eval_results)
trainer.save_metrics("eval", eval_results)

In [ ]:
# ============================================================
# SAVE LORA ADAPTER
# Saves only adapter weights (~100-300MB)
# NOT the full 12B base model
# ============================================================
import os as _os

ADAPTER_DIR = f"{OUTPUT_DIR}/final-adapter"

trainer.model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)

print(f"✅ Adapter saved to: {ADAPTER_DIR}")
print("\n── Saved files ────────────────────────────────────────")
total_mb = 0
for fname in sorted(_os.listdir(ADAPTER_DIR)):
    fpath = f"{ADAPTER_DIR}/{fname}"
    if _os.path.isfile(fpath):
        mb = _os.path.getsize(fpath) / 1e6
        total_mb += mb
        print(f"   {fname:45s} {mb:8.1f} MB")
print(f"   {'─'*55}")
print(f"   {'TOTAL':45s} {total_mb:8.1f} MB")

In [ ]:
# ============================================================
#  QUALITATIVE INFERENCE TEST
# ============================================================

trainer.model.config.use_cache = True
trainer.model.eval()
gc.collect()
torch.cuda.empty_cache()
vram("inference start")

def generate_answer(question: str, max_new_tokens: int = 350) -> str:
    """Generate bioinformatics answer using fine-tuned model."""
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": question},
    ]
    prompt = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True,
    )
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=MAX_SEQ_LEN,
    ).to("cuda:0")

    with torch.inference_mode():
        output_ids = trainer.model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.3,
            top_p=0.9,
            repetition_penalty=1.1,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    new_ids = output_ids[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_ids, skip_special_tokens=True).strip()

# Test questions
test_questions = [
    "What is the difference between BLAST and HMMER?",
    "Explain Phred quality scores in next-generation sequencing.",
    "How does GATK HaplotypeCaller detect SNPs and indels?",
    "What are CpG islands and why are they biologically important?",
    "Describe the main steps in a standard RNA-seq analysis pipeline.",
]

print("=" * 70)
print("🔬 QUALITATIVE EVALUATION — Gemma-4 Bioinformatics")
print("=" * 70)
for i, q in enumerate(test_questions, 1):
    print(f"\nQ{i}: {q}")
    print("-" * 70)
    print(f"A:  {generate_answer(q)}")

vram("inference end")

In [ ]:
# ============================================================
# MERGE LoRA ADAPTER INTO FULL BASE MODEL
# Run this AFTER Cell 13 saved: ./gemma4-12b-bioinfo-qlora/final-adapter
# ============================================================
HF_TOKEN     = "hf_XXXXXXXXXXXXXXXXXXXXXXXXXXXXXX"   # ← your token

import os, gc, torch
from transformers import AutoTokenizer, AutoModelForImageTextToText
from peft import PeftModel

# ── Paths from your notebook ────────────────────────────────
MODEL_ID    = "google/gemma-4-12B-it"
ADAPTER_DIR = "./gemma4-12b-bioinfo-qlora/final-adapter"
MERGED_DIR  = "./gemma4-12b-bioinfo-merged"

# Use a separate repo name for the merged/full model

# Set True only when you want to upload to Hugging Face Hub
PUSH_MERGED_TO_HUB = False

# ── Cleanup old training objects to free VRAM/RAM ───────────
for obj in ["trainer", "model", "base_model", "peft_model", "merged_model"]:
    if obj in globals():
        del globals()[obj]

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

print("🔁 Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(
    ADAPTER_DIR,
    token=HF_TOKEN,
    trust_remote_code=True,
)

# IMPORTANT:
# Do NOT merge into the 4-bit QLoRA training model.
# Reload the base model in bf16, attach adapter, then merge.
print("📥 Loading base model in bf16 for clean merge...")

base_model = AutoModelForImageTextToText.from_pretrained(
    MODEL_ID,
    token=HF_TOKEN,
    dtype=torch.bfloat16,
    device_map={"": "cpu"},          # safest for merge; avoids 24GB GPU OOM
    attn_implementation="eager",     # required by your Gemma-4 setup
    low_cpu_mem_usage=True,
    trust_remote_code=True,
)

print("🔌 Loading LoRA adapter...")
peft_model = PeftModel.from_pretrained(
    base_model,
    ADAPTER_DIR,
    is_trainable=False,
)

peft_model.eval()

print("🧬 Merging adapter into base model...")
merged_model = peft_model.merge_and_unload(safe_merge=True)
merged_model.eval()

# Safety check: merged model should no longer be a PEFT model
assert not hasattr(merged_model, "peft_config"), "❌ Merge failed: PEFT adapter still attached."

print(f"💾 Saving merged model to: {MERGED_DIR}")

merged_model.save_pretrained(
    MERGED_DIR,
    safe_serialization=True,
    max_shard_size="1900MB",   # GitHub LFS Free/Pro per-file safe shard size
)

tokenizer.save_pretrained(MERGED_DIR)

print("✅ Merged model saved successfully.")
print("\n── Saved merged files ────────────────────────────────")
total_gb = 0
for fname in sorted(os.listdir(MERGED_DIR)):
    fpath = os.path.join(MERGED_DIR, fname)
    if os.path.isfile(fpath):
        size_gb = os.path.getsize(fpath) / 1024**3
        total_gb += size_gb
        print(f"{fname:55s} {size_gb:8.2f} GB")
print(f"{'TOTAL':55s} {total_gb:8.2f} GB")

In [ ]:
# ============================================================
#  FULLY OPTIMIZED INFERENCE (GPU 0 + 4-BIT + ANTI-LOOPING)
# ============================================================

import gc, re, torch
from transformers import AutoTokenizer, AutoModelForImageTextToText, BitsAndBytesConfig

MERGED_DIR = "./gemma4-12b-bioinfo-merged"

SYSTEM_PROMPT = (
    "You are an expert bioinformatics assistant with deep knowledge of "
    "genomics, proteomics, transcriptomics, sequence analysis, "
    "biological databases (UniProt, NCBI, Ensembl, PDB), and tools "
    "(BLAST, HMMER, BWA, GATK, DESeq2, Bowtie2, STAR, Samtools, BioPython). "
    "Provide accurate, concise, and scientifically rigorous answers."
)

# ── 1. Clean Memory ─────────────────────────────────────────
for obj in ["trainer", "base_model", "peft_model", "merged_model"]:
    if obj in globals():
        del globals()[obj]

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

# ── 2. Quantization Config ──────────────────────────────────
# Shrinks the model down to ~8GB so it easily fits on GPU 0
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16, 
)

# ── 3. Load Tokenizer & Model ───────────────────────────────
print("🔁 Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(
    MERGED_DIR,
    trust_remote_code=True,
)

print("🧠 Loading merged model in 4-bit strictly onto GPU 0...")
model = AutoModelForImageTextToText.from_pretrained(
    MERGED_DIR,
    device_map={"": 0},                 
    quantization_config=quant_config,   
    attn_implementation="eager",
    trust_remote_code=True,
)

model.eval()
model.config.use_cache = True

# ── 4. Helper Functions ─────────────────────────────────────
def clean_gemma4_response(text: str) -> str:
    """Strips out Gemma 4's internal thought channels, loops, and special tokens."""
    # 1. Remove Gemma 4 thought block if it appears
    text = re.sub(
        r"<\|channel\>thought.*?<channel\|>",
        "",
        text,
        flags=re.DOTALL,
    )
    
    # 2. Aggressive truncation to stop hallucinated conversation loops
    if "<|turn>" in text:
        text = text.split("<|turn>")[0]
    if "<turn|>" in text:
        text = text.split("<turn|>")[0]
        
    # 3. Clean up any remaining typical markers
    text = text.replace("<eos>", "").replace("<bos>", "")
    
    return text.strip()

def ask_bioinfo(question: str, max_new_tokens: int = 512, debug: bool = True):
    """Formats the prompt, generates a response, and cleans the output."""
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": question.strip()},
    ]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False, 
    )

    if debug:
        print("\n🔍 RAW PROMPT:")
        print(prompt)
        print("-" * 60)

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    # Use inference_mode for maximum speed
    with torch.inference_mode():
        print("✨ Generating response...\n")
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,          
            temperature=0.2,         
            top_p=0.9,
            repetition_penalty=1.15,  # Stops the model from looping phrases
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    input_len = inputs["input_ids"].shape[-1]
    new_tokens = outputs[0][input_len:]

    raw = tokenizer.decode(new_tokens, skip_special_tokens=False)
    clean = clean_gemma4_response(raw)

    print("🤖 MODEL RESPONSE:")
    print(clean)
    print("========================================================\n")
    return clean

# ── 5. Test the Model ───────────────────────────────────────
ask_bioinfo("What is BLAST used for in bioinformatics?")

In [ ]:
# ============================================================
#  ASK YOUR OWN BIOINFORMATICS QUESTIONS
# ============================================================

# 1. Type your question here
my_question = (
    "Explain the difference between local and global sequence alignment. "
    "When should I use Smith-Waterman versus Needleman-Wunsch?"
)

# 2. Get the response
# Note: I set debug=False here so it hides the raw prompt and just gives you the answer!
print("🧠 Asking the model...")
response = ask_bioinfo(
    question=my_question, 
    max_new_tokens=512, 
    debug=False  
)

In [ ]:
# ============================================================
# SETUP LLAMA.CPP (PATH-ROBUST BUILDER)
# ============================================================
import os
import sys
import subprocess

# 1. Dynamically locate the cmake executable inside your gemma4 environment
env_bin_dir = os.path.dirname(sys.executable)
cmake_path = os.path.join(env_bin_dir, "cmake")

if not os.path.exists(cmake_path):
    raise FileNotFoundError(f"Could not find cmake at {cmake_path}. Please re-run the %pip install cell.")

print(f"🛠️ Found local CMake at: {cmake_path}")
print("📦 Generating build configuration...")

# 2. Run CMake configure inside the llama.cpp directory
configure_cmd = [cmake_path, "-B", "build"]
result_config = subprocess.run(configure_cmd, cwd="llama.cpp", capture_output=True, text=True)

if result_config.returncode != 0:
    print("❌ Configuration failed. Error log:")
    print(result_config.stderr)
else:
    print("✅ Build files generated successfully. Compiling llama.cpp (this takes a minute)...")
    
    # 3. Run CMake build
    build_cmd = [cmake_path, "--build", "build", "--config", "Release", "-j", "8"]
    result_build = subprocess.run(build_cmd, cwd="llama.cpp", capture_output=True, text=True)
    
    if result_build.returncode != 0:
        print("❌ Compilation failed. Error log:")
        print(result_build.stderr)
    else:
        print("🎉 Successfully compiled llama.cpp! You are ready for Cell 19.")

In [ ]:
# ============================================================
#  UPDATE LLAMA.CPP FOR GEMMA 4 SUPPORT
# ============================================================
import os, sys

print("🔄 Pulling the bleeding-edge master branch...")
!cd llama.cpp && git pull

print("\n📦 Recompiling llama.cpp with Gemma 4 Unified support...")
cmake_path = os.path.join(os.path.dirname(sys.executable), "cmake")
!cd llama.cpp && {cmake_path} --build build --config Release -j 8

print("\n✅ Update complete! You are ready to convert.")

In [ ]:
# ============================================================
#  CONVERT HUGGING FACE TO GGUF (BF16)
# ============================================================
import sys

!{sys.executable} llama.cpp/convert_hf_to_gguf.py ./gemma4-12b-bioinfo-merged \
  --outfile ./gemma4-12b-bioinfo-BF16.gguf \
  --outtype bf16

In [ ]:
# ============================================================
#  QUANTIZE TO 4-BIT (Q4_K_M)
# ============================================================
!./llama.cpp/build/bin/llama-quantize ./gemma4-12b-bioinfo-BF16.gguf ./gemma4-12b-bioinfo-Q4_K_M.gguf Q4_K_M

In [ ]:
# ============================================================
#  TEST 4-BIT GGUF & BENCHMARK SPEED
# ============================================================
import os

# 1. Define your question (using the Gemma 4 prompt structure)
question = "Explain the role of CRISPR-Cas9 in genome editing in two concise sentences."
prompt_text = f"<bos><|turn>user\n{question}<|turn>model\n"

# 2. Save to a temporary file to avoid bash syntax errors
with open("temp_prompt.txt", "w", encoding="utf-8") as f:
    f.write(prompt_text)

# 3. Execute llama-cli with full GPU offloading
print("🚀 Booting up llama.cpp and generating response...\n" + "="*50 + "\n")

!./llama.cpp/build/bin/llama-cli \
  -m ./gemma4-12b-bioinfo-Q4_K_M.gguf \
  -f temp_prompt.txt \
  -n 512 \
  -c 1024 \
  --temp 0.2 \
  -ngl 99

In [ ]:
# ============================================================
# TEST LOCAL GGUF MODEL IN JUPYTER — 3 QUESTIONS
# Uses llama-cpp-python
# Prompt format fixed for Gemma 4 GGUF: NO <bos>
# ============================================================

import os
import sys
import subprocess
from pathlib import Path

# Install llama-cpp-python if missing
try:
    from llama_cpp import Llama
except ImportError:
    print("Installing llama-cpp-python...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-U", "llama-cpp-python"])
    from llama_cpp import Llama

# ── Set your GGUF path here ─────────────────────────────────
# Example:
# GGUF_MODEL_PATH = "./gemma4-12b-bioinfo-Q4_K_M.gguf"

GGUF_MODEL_PATH = "./gemma4-12b-bioinfo-Q4_K_M.gguf"

# Auto-find a GGUF file if the path above does not exist
if not Path(GGUF_MODEL_PATH).exists():
    gguf_files = list(Path(".").rglob("*.gguf"))
    if len(gguf_files) == 0:
        raise FileNotFoundError(
            "No .gguf file found. Put your GGUF model in this notebook folder "
            "or update GGUF_MODEL_PATH."
        )
    GGUF_MODEL_PATH = str(gguf_files[0])
    print(f"Using found GGUF model: {GGUF_MODEL_PATH}")

print(f"Loading GGUF model: {GGUF_MODEL_PATH}")

llm = Llama(
    model_path=GGUF_MODEL_PATH,
    n_ctx=2048,
    n_gpu_layers=-1,   # use GPU if available; set 0 for CPU only
    verbose=False,
)

system_prompt = (
    "You are an expert bioinformatics assistant with deep knowledge of genomics, "
    "proteomics, transcriptomics, sequence analysis, biological databases, and "
    "bioinformatics tools. Provide accurate, concise, scientifically rigorous answers."
)

questions = [
    "Explain the difference between local and global sequence alignment.",
    "What is the purpose of BLAST in bioinformatics?",
    "Explain how CRISPR-Cas9 is used in genome editing.",
]

def make_prompt(question: str) -> str:
    return (
        f"<|turn>user\n"
        f"{system_prompt}\n\n"
        f"Question: {question}\n"
        f"<|turn>model\n"
    )

for i, question in enumerate(questions, start=1):
    prompt = make_prompt(question)

    output = llm(
        prompt,
        max_tokens=512,
        temperature=0.2,
        top_p=0.9,
        repeat_penalty=1.1,
        stop=["<|turn>user", "<eos>"],
        echo=False,
    )

    answer = output["choices"][0]["text"].strip()

    print("=" * 80)
    print(f"QUESTION {i}: {question}")
    print("-" * 80)
    print(answer)
    print()